# LeetCode 486 — Predict the Winner bằng Python

Link đề: https://leetcode.com/problems/predict-the-winner/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, Minimax, Game Theory (trò chơi tổng bằng không), Recursion + Memoization

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def predictTheWinner(self, nums: List[int]) -> bool:
        ...
```

Ghi chú: bài này không Premium. Đề được viết lại bằng tiếng Việt theo đề gốc; các ví dụ phụ do notebook tự dựng sẽ được ghi rõ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai người chơi lần lượt lấy số ở **một trong hai đầu** của mảng; cả hai đều chơi **tối ưu**; hỏi người chơi 1 có thắng (hoặc hòa) không.
2. Dùng được các công cụ Python: đệ quy, `functools.lru_cache`/`cache`, bảng hai chiều `[[0] * n for _ in range(n)]`, `max`/`min`, và hiểu vì sao phải tránh `[[0] * n] * n`.
3. Nắm ý tưởng cốt lõi của trò chơi hai người **tổng bằng không**: thay vì theo dõi hai điểm, theo dõi **hiệu số điểm** `dfs(i, j)` = (điểm người đang đi) − (điểm đối thủ) và dùng công thức `nums[i] - dfs(i+1, j)`.
4. Cài đặt 5 cách khác nhau về ý tưởng: **đệ quy minimax thuần theo hai điểm số → đệ quy có nhớ theo hiệu số → quy hoạch động bảng 2D → bảng 1D cuốn chiếu → mẹo số chẵn (người 1 luôn không thua) + DP cho `n` lẻ**.
5. Tránh các lỗi kinh điển: tham lam "luôn lấy số lớn hơn", đảo dấu sai ở công thức hiệu số, nhầm `>= 0` với `> 0` (hòa vẫn thắng), và khởi tạo bảng sai.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Hai người chơi: **Người 1** và **Người 2**. Người 1 đi **trước**. Cả hai bắt đầu với 0 điểm.

Ở mỗi lượt, người chơi lấy một số ở **một trong hai đầu** mảng (`nums[0]` hoặc `nums[-1]`), số đó bị **xóa** khỏi mảng và được cộng vào điểm của người đó. Trò chơi kết thúc khi mảng rỗng.

Trả về `True` nếu điểm của Người 1 **lớn hơn hoặc bằng** điểm của Người 2, ngược lại trả về `False`. Hòa tính là Người 1 thắng. Giả sử cả hai chơi **tối ưu**.

**Ví dụ 1:** `nums = [1, 5, 2]` → `False`. Người 1 lấy 1 hoặc 2 đều dẫn đến kết quả như nhau: Người 2 lấy `5`; Người 1 chỉ được tổng `1 + 2 = 3` trong khi Người 2 có `5`.

**Ví dụ 2:** `nums = [1, 5, 233, 7]` → `True`. Người 1 lấy `1`, còn `[5, 233, 7]`; Người 2 phải lấy `5` hoặc `7`, và dù lấy cái nào Người 1 cũng lấy được `233` ở lượt sau. Người 1 có `234` điểm, Người 2 có `12`.

**Ràng buộc:** `1 <= nums.length <= 20`; `0 <= nums[i] <= 10^7`.

In [1]:
vi_du = [([1, 5, 2], False), ([1, 5, 233, 7], True)]
for nums, kq in vi_du:
    print(f"nums = {nums} -> kỳ vọng {kq}")

nums = [1, 5, 2] -> kỳ vọng False
nums = [1, 5, 233, 7] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Đệ quy và trò chơi "lấy hai đầu"

Mỗi lượt đi, mảng còn lại luôn là một **đoạn liên tiếp** `nums[i..j]` của mảng gốc (vì chỉ lấy từ hai đầu). Vậy trạng thái trò chơi chỉ cần hai số `(i, j)`, cộng với việc biết lượt của ai. Đây là ý tưởng quan trọng: một "bức tranh" của trò chơi gọn trong hai chỉ số.

In [2]:
nums = [1, 5, 2]
# lượt đầu có hai lựa chọn, dẫn tới hai đoạn con
print("lấy đầu trái ->", nums[1:])    # đoạn [1..2]
print("lấy đầu phải ->", nums[:-1])   # đoạn [0..1]

lấy đầu trái -> [5, 2]
lấy đầu phải -> [1, 5]


### 3.2. Tối ưu = Minimax, trò chơi "tổng bằng không"

Mỗi người muốn điểm của mình **cao nhất**. Vì tổng điểm của hai người là cố định (tổng của mảng), **tăng điểm của mình** cũng đồng nghĩa với **tăng hiệu số điểm (mình − đối thủ)**. Do đó mục tiêu của cả hai quy về: người đang đi muốn **tối đa hóa hiệu số**; người kia muốn **tối thiểu hóa** hiệu số đó (từ góc nhìn của người đầu).

Cụ thể: nếu Người 1 muốn tối đa hóa `(điểm 1 − điểm 2)` thì Người 2 muốn tối thiểu hóa chính biểu thức đó (vì đó là tối đa hóa `điểm 2 − điểm 1`). Đó là **minimax**: một bên `max`, một bên `min`.

In [3]:
tong = sum([1, 5, 2])
d1, d2 = 3, 5
print("tổng", tong, "| điểm 1 + điểm 2 =", d1 + d2)       # cố định
print("hiệu số P1 - P2 =", d1 - d2)

tổng 8 | điểm 1 + điểm 2 = 8
hiệu số P1 - P2 = -2


### 3.3. Mẹo "hiệu số góc nhìn người đang đi"

Thay vì lưu cả hai điểm, định nghĩa `dfs(i, j)` = **hiệu số tối đa** (điểm của **người đang đi** − điểm của **đối thủ**) mà người đang đi đạt được trên đoạn `nums[i..j]`, nếu cả hai chơi tối ưu.

- Nếu lấy đầu trái `nums[i]`: người này +`nums[i]`, rồi đối thủ trở thành "người đang đi" trên đoạn `[i+1..j]` và đạt hiệu số `dfs(i+1, j)` (góc nhìn đối thủ). Từ góc nhìn người này, hiệu số là `nums[i] - dfs(i+1, j)`.
- Tương tự đầu phải: `nums[j] - dfs(i, j-1)`.
- Chọn cái lớn hơn: `dfs(i, j) = max(nums[i] - dfs(i+1, j), nums[j] - dfs(i, j-1))`.
- Đoạn rỗng (`i > j`): `0`.

Người 1 thắng khi `dfs(0, n-1) >= 0`. Dấu **trừ** là điểm mấu chốt: nó đổi góc nhìn mỗi lượt.

In [4]:
# Thử tay với [1, 5]: dfs(0,1) = max(1 - dfs(1,1), 5 - dfs(0,0)), dfs(i,i) = nums[i]
dfs_11, dfs_00 = 5, 1
print("dfs(0,1) =", max(1 - dfs_11, 5 - dfs_00))     # max(-4, 4) = 4  -> lấy 5, được hơn 4 điểm

dfs(0,1) = 4


### 3.4. Ghi nhớ kết quả (memoization) với `functools.lru_cache`

Cùng một đoạn `(i, j)` có thể được truy cập nhiều lần (từ nhiều lịch sử khác nhau). `@lru_cache(maxsize=None)` (hoặc `@cache` từ Python 3.9) **ghi nhớ** kết quả theo tham số: lần sau gọi lại cùng tham số thì trả luôn, không tính lại.

In [5]:
from functools import lru_cache

dem_goi = 0

@lru_cache(maxsize=None)
def fib(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30), "số lần thân hàm thực sự chạy:", dem_goi)   # 31, không phải ~2 triệu
print(fib.cache_info())

832040 số lần thân hàm thực sự chạy: 31
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


### 3.5. Bảng 2 chiều và cái bẫy `[[0] * n] * n`

`[[0] * n for _ in range(n)]` tạo `n` hàng **độc lập**. Còn `[[0] * n] * n` tạo `n` tham chiếu tới **cùng một** hàng: sửa một ô là sửa cả cột. Hãy xem bằng mắt.

In [6]:
dung = [[0] * 3 for _ in range(3)]
sai = [[0] * 3] * 3
dung[0][0] = 9
sai[0][0] = 9
print("đúng:", dung)     # chỉ hàng 0 đổi
print("sai :", sai)      # cả ba hàng cùng đổi

đúng: [[9, 0, 0], [0, 0, 0], [0, 0, 0]]
sai : [[9, 0, 0], [9, 0, 0], [9, 0, 0]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Tham lam có đủ không?** Thử "luôn lấy đầu lớn hơn" với `[1, 5, 233, 7]`: Người 1 lấy `7` (so `1` và `7`), Người 2 gặp `[1, 5, 233]` lấy `233`, Người 1 lấy `5`, Người 2 lấy `1`. P1 = `12`, P2 = `234`. Thua! Trong khi đáp án đúng là Người 1 thắng nếu **lấy `1` trước** (hy sinh cái nhỏ để không mở đường cho `233`). Vậy tham lam sai, phải xét hết khả năng.

**Tính tay `[1, 5, 2]`:**

- Hiệu số mỗi đoạn một phần tử: `dfs(0,0) = 1`, `dfs(1,1) = 5`, `dfs(2,2) = 2`.
- Đoạn hai phần tử: `dfs(0,1) = max(1 - 5, 5 - 1) = 4`; `dfs(1,2) = max(5 - 2, 2 - 5) = 3`.
- Đoạn ba phần tử: `dfs(0,2) = max(nums[0] - dfs(1,2), nums[2] - dfs(0,1)) = max(1 - 3, 2 - 4) = -2`.

`-2 < 0` ⇒ Người 1 thua (điểm P1 = `3`, P2 = `5`, hiệu `-2`) ⇒ `False`. ✓.

**Tính tay `[1, 5, 233, 7]`:** `dfs(0,3) = 222 >= 0` ⇒ `True` (sẽ tính chi tiết ở mục 8).

In [7]:
nums = [1, 5, 2]
d = {}
for i in range(3):
    d[(i, i)] = nums[i]
d[(0, 1)] = max(nums[0] - d[(1, 1)], nums[1] - d[(0, 0)])
d[(1, 2)] = max(nums[1] - d[(2, 2)], nums[2] - d[(1, 1)])
d[(0, 2)] = max(nums[0] - d[(1, 2)], nums[2] - d[(0, 1)])
print(d)
print("Người 1 thắng?", d[(0, 2)] >= 0)

{(0, 0): 1, (1, 1): 5, (2, 2): 2, (0, 1): 4, (1, 2): 3, (0, 2): -2}
Người 1 thắng? False


## 5. Cách 1 — Brute Force / baseline: minimax thuần, theo dõi cả hai điểm số

**Ý tưởng:** mô phỏng đúng đề: hàm `tran(l, r, diem1, diem2, luot)` đi hết mọi cách chơi. Người 1 chọn nước đi **tối đa hóa** hiệu số cuối `diem1 - diem2`, người 2 chọn **tối thiểu hóa** nó. Kết thúc khi `l > r`: trả `diem1 - diem2`.

Cách này **độc lập** với mẹo "hiệu số góc nhìn người đang đi" (ta vẫn tin vào định nghĩa, theo dõi hai điểm rõ ràng), nên rất hợp làm **oracle**.

In [8]:
def chenh_lech_vet_can(nums, dem=None):
    n = len(nums)

    def tran(l, r, diem1, diem2, luot):
        if dem is not None:
            dem[0] += 1
        if l > r:
            return diem1 - diem2                   # hiệu số cuối cùng (góc nhìn Người 1)
        if luot == 1:                              # Người 1: tối đa hóa
            return max(tran(l + 1, r, diem1 + nums[l], diem2, 2),
                       tran(l, r - 1, diem1 + nums[r], diem2, 2))
        return min(tran(l + 1, r, diem1, diem2 + nums[l], 1),       # Người 2: tối thiểu hóa hiệu số
                   tran(l, r - 1, diem1, diem2 + nums[r], 1))

    return tran(0, n - 1, 0, 0, 1)

def thang_vet_can(nums):
    return chenh_lech_vet_can(nums) >= 0

for nums, kq in vi_du:
    print(nums, "->", chenh_lech_vet_can(nums), thang_vet_can(nums))
    assert thang_vet_can(nums) == kq

[1, 5, 2] -> -2 False
[1, 5, 233, 7] -> 222 True


**Độ phức tạp:** mỗi lượt có 2 lựa chọn, `n` lượt ⇒ cây gọi có `~2^(n+1)` nút: thời gian `O(2^n)`, bộ nhớ `O(n)` (độ sâu đệ quy). Với `n = 20` là cỡ `2 triệu` lần gọi: được nhưng chậm.

**Nút thắt:** số lần gọi tăng gấp đôi mỗi khi thêm một phần tử. Hãy đếm:

In [9]:
for n in (6, 10, 14, 18):
    dem = [0]
    chenh_lech_vet_can(list(range(1, n + 1)), dem)
    print(f"n = {n:>2}: {dem[0]:>8} lần gọi (2^(n+1) - 1 = {2 ** (n + 1) - 1})")

n =  6:      127 lần gọi (2^(n+1) - 1 = 127)
n = 10:     2047 lần gọi (2^(n+1) - 1 = 2047)
n = 14:    32767 lần gọi (2^(n+1) - 1 = 32767)
n = 18:   524287 lần gọi (2^(n+1) - 1 = 524287)


Nhưng số **đoạn khác nhau** `(l, r)` chỉ có `n(n+1)/2` (với `n = 20` là 210). Cùng một đoạn bị tính lại rất nhiều lần, với nhiều giá trị `(diem1, diem2)` khác nhau, nhưng **hiệu số tối ưu từ đoạn đó trở đi thì không phụ thuộc** vào điểm đã có. Đó là bước đột phá.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Trạng thái chỉ cần `(i, j)`:** điểm đã tích lũy không ảnh hưởng đến quyết định trong tương lai; cái còn ảnh hưởng chỉ là đoạn `nums[i..j]` còn lại.
2. **Quy về hiệu số:** `dfs(i, j)` = hiệu số tối đa (người đang đi − đối thủ) trên đoạn `[i..j]`; công thức
   `dfs(i, j) = max(nums[i] - dfs(i+1, j), nums[j] - dfs(i, j-1))`, `dfs(i, j) = 0` khi `i > j`.
3. **Số trạng thái** `O(n²)`, mỗi trạng thái `O(1)` ⇒ `O(n²)`.
4. **Đáp án:** `dfs(0, n-1) >= 0`.
5. **Có thể làm bottom-up:** `dp[i][j]` phụ thuộc `dp[i+1][j]` và `dp[i][j-1]` (đoạn ngắn hơn) ⇒ tính theo độ dài tăng dần, hoặc `i` giảm dần.
6. **Tối ưu bộ nhớ** vì hàng `i` chỉ cần hàng `i+1` ⇒ bảng 1D.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ theo hiệu số (top-down)

**Giải thích từng đoạn:**

- `@lru_cache(maxsize=None)` bọc hàm `dfs(i, j)`.
- `if i > j: return 0` (hết số).
- `return max(nums[i] - dfs(i+1, j), nums[j] - dfs(i, j-1))`.
- Hàm `dfs` định nghĩa **bên trong** hàm chính để truy cập `nums` (mỗi lần gọi hàm chính có bộ nhớ đệm mới).

In [10]:
def chenh_lech_memo(nums):
    n = len(nums)

    @lru_cache(maxsize=None)
    def dfs(i, j):
        if i > j:
            return 0
        return max(nums[i] - dfs(i + 1, j), nums[j] - dfs(i, j - 1))

    kq = dfs(0, n - 1)
    return kq, dfs.cache_info().currsize

def thang_memo(nums):
    return chenh_lech_memo(nums)[0] >= 0

for nums, kq in vi_du:
    print(nums, "->", chenh_lech_memo(nums), thang_memo(nums))
    assert thang_memo(nums) == kq
print("n = 20: số đoạn đã tính =", chenh_lech_memo(list(range(1, 21)))[1], "(n(n+1)/2 + ... đoạn rỗng)")

[1, 5, 2] -> (-2, 10) False
[1, 5, 233, 7] -> (222, 15) True
n = 20: số đoạn đã tính = 231 (n(n+1)/2 + ... đoạn rỗng)


**Code nộp LeetCode (Cách 2):**

```python
from functools import cache
from typing import List

class Solution:
    def predictTheWinner(self, nums: List[int]) -> bool:
        @cache
        def dfs(i: int, j: int) -> int:
            if i > j:
                return 0
            return max(nums[i] - dfs(i + 1, j), nums[j] - dfs(i, j - 1))

        return dfs(0, len(nums) - 1) >= 0
```

**Độ phức tạp:** `O(n²)` thời gian (mỗi trạng thái `O(1)`), `O(n²)` bộ nhớ cho bộ nhớ đệm và `O(n)` đệ quy.

### Cách 3 — Quy hoạch động bottom-up, bảng 2D

**Ý tưởng:** biến công thức thành bảng `dp[i][j]`. Đoạn dài 1: `dp[i][i] = nums[i]`. Đoạn dài `L = 2..n`: `dp[i][j] = max(nums[i] - dp[i+1][j], nums[j] - dp[i][j-1])` với `j = i + L - 1`. Vì `dp[i+1][j]` và `dp[i][j-1]` thuộc đoạn **ngắn hơn một đơn vị** nên khi duyệt theo độ dài tăng dần, chúng đã sẵn sàng.

**Giải thích từng đoạn:** (1) tạo bảng `n × n` toàn 0 (nhớ cách tạo đúng); (2) điền đường chéo `dp[i][i]`; (3) hai vòng lặp `L`, `i`; (4) trả `dp[0][n-1] >= 0`.

In [11]:
def chenh_lech_dp2d(nums):
    n = len(nums)
    dp = [[0] * n for _ in range(n)]
    for i in range(n):
        dp[i][i] = nums[i]
    for L in range(2, n + 1):
        for i in range(0, n - L + 1):
            j = i + L - 1
            dp[i][j] = max(nums[i] - dp[i + 1][j], nums[j] - dp[i][j - 1])
    return dp[0][n - 1]

def thang_dp2d(nums):
    return chenh_lech_dp2d(nums) >= 0

for nums, kq in vi_du:
    print(nums, "->", chenh_lech_dp2d(nums), thang_dp2d(nums))
    assert thang_dp2d(nums) == kq

[1, 5, 2] -> -2 False
[1, 5, 233, 7] -> 222 True


**Code nộp LeetCode (Cách 3):**

```python
from typing import List

class Solution:
    def predictTheWinner(self, nums: List[int]) -> bool:
        n = len(nums)
        dp = [[0] * n for _ in range(n)]
        for i in range(n):
            dp[i][i] = nums[i]
        for L in range(2, n + 1):
            for i in range(n - L + 1):
                j = i + L - 1
                dp[i][j] = max(nums[i] - dp[i + 1][j], nums[j] - dp[i][j - 1])
        return dp[0][n - 1] >= 0
```

**Độ phức tạp:** `O(n²)` thời gian, `O(n²)` bộ nhớ.

### Cách 4 — Bảng 1D cuốn chiếu (rolling array)

**Ý tưởng:** nếu duyệt `i` từ `n-1` xuống `0`, hàng `i` chỉ phụ thuộc hàng `i+1` và chính hàng `i` ở cột `j-1`. Dùng **một** mảng `dp` độ dài `n`, trong đó `dp[j]` lần lượt đại diện `dp[i+1][j]` (trước khi cập nhật) rồi `dp[i][j]` (sau cập nhật).

**Giải thích từng đoạn:**

- Khởi tạo `dp = nums[:]` (đại diện hàng `i = n-1`, trong đó `dp[j] = dp[j][j]` chỉ đúng ở `j = i`, nhưng ta chỉ dùng `dp[i]` ở hàng `i`).
- Với `i` từ `n-2` xuống `0`, với `j` từ `i+1` đến `n-1`: `dp[j] = max(nums[i] - dp[j], nums[j] - dp[j-1])`. Ở đây `dp[j]` (vế phải) là `dp[i+1][j]` cũ, còn `dp[j-1]` là `dp[i][j-1]` vừa cập nhật (với `j - 1 = i` thì chính là `nums[i]` ban đầu).
- Cuối cùng `dp[n-1]` là `dp[0][n-1]`.

In [12]:
def chenh_lech_dp1d(nums):
    n = len(nums)
    dp = nums[:]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            dp[j] = max(nums[i] - dp[j], nums[j] - dp[j - 1])
    return dp[n - 1]

def thang_dp1d(nums):
    return chenh_lech_dp1d(nums) >= 0

for nums, kq in vi_du:
    print(nums, "->", chenh_lech_dp1d(nums), thang_dp1d(nums))
    assert thang_dp1d(nums) == kq

[1, 5, 2] -> -2 False
[1, 5, 233, 7] -> 222 True


**Code nộp LeetCode (Cách 4):**

```python
from typing import List

class Solution:
    def predictTheWinner(self, nums: List[int]) -> bool:
        n = len(nums)
        dp = nums[:]
        for i in range(n - 2, -1, -1):
            for j in range(i + 1, n):
                dp[j] = max(nums[i] - dp[j], nums[j] - dp[j - 1])
        return dp[n - 1] >= 0
```

**Độ phức tạp:** `O(n²)` thời gian, `O(n)` bộ nhớ.

### Cách 5 — Mẹo số chẵn: nếu `n` chẵn thì Người 1 luôn không thua; `n` lẻ thì DP

**Quan sát (chứng minh ở mục 10):** khi số phần tử `n` **chẵn**, Người 1 luôn có chiến lược **đảm bảo lấy hết các phần tử ở chỉ số chẵn** hoặc **hết các phần tử ở chỉ số lẻ** — tùy bên nào có tổng lớn hơn. Vì thế điểm của Người 1 luôn `≥` điểm Người 2 ⇒ trả `True` ngay, không cần tính toán. Với `n` lẻ ta dùng DP như thường.

**Giải thích từng đoạn:** `if len(nums) % 2 == 0: return True`; ngược lại gọi Cách 4.

In [13]:
def thang_chan_le(nums):
    if len(nums) % 2 == 0:
        return True
    return thang_dp1d(nums)

for nums, kq in vi_du:
    print(nums, "->", thang_chan_le(nums))
    assert thang_chan_le(nums) == kq
print("ví dụ n lẻ:", thang_chan_le([1, 5, 2]), "| n chẵn luôn True:", thang_chan_le([1, 100, 2, 3, 4, 5]))

[1, 5, 2] -> False
[1, 5, 233, 7] -> True
ví dụ n lẻ: False | n chẵn luôn True: True


**Code nộp LeetCode (Cách 5):**

```python
from typing import List

class Solution:
    def predictTheWinner(self, nums: List[int]) -> bool:
        n = len(nums)
        if n % 2 == 0:
            return True
        dp = nums[:]
        for i in range(n - 2, -1, -1):
            for j in range(i + 1, n):
                dp[j] = max(nums[i] - dp[j], nums[j] - dp[j - 1])
        return dp[n - 1] >= 0
```

**Độ phức tạp:** `n` chẵn `O(1)`; `n` lẻ `O(n²)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Cách 3 (bảng 2D) với `nums = [1, 5, 233, 7]` (`n = 4`). Ta điền theo độ dài `L` tăng dần.

| L | (i, j) | công thức | giá trị |
|---|---|---|---|
| 1 | (0,0),(1,1),(2,2),(3,3) | `nums[i]` | 1, 5, 233, 7 |
| 2 | (0,1) | max(1 − 5, 5 − 1) | **4** |
| 2 | (1,2) | max(5 − 233, 233 − 5) | **228** |
| 2 | (2,3) | max(233 − 7, 7 − 233) | **226** |
| 3 | (0,2) | max(1 − dp[1][2], 233 − dp[0][1]) = max(1 − 228, 233 − 4) | **229** |
| 3 | (1,3) | max(5 − dp[2][3], 7 − dp[1][2]) = max(5 − 226, 7 − 228) | **−221** |
| 4 | (0,3) | max(1 − dp[1][3], 7 − dp[0][2]) = max(1 + 221, 7 − 229) | **222** |

`dp[0][3] = 222 >= 0` ⇒ `True`. Hiệu số `222` khớp với `234 − 12`.

In [14]:
def trace_dp2d(nums):
    n = len(nums)
    dp = [[0] * n for _ in range(n)]
    for i in range(n):
        dp[i][i] = nums[i]
    print("L=1: đường chéo =", [dp[i][i] for i in range(n)])
    for L in range(2, n + 1):
        for i in range(0, n - L + 1):
            j = i + L - 1
            lay_trai = nums[i] - dp[i + 1][j]
            lay_phai = nums[j] - dp[i][j - 1]
            dp[i][j] = max(lay_trai, lay_phai)
            nuoc = "trái" if lay_trai >= lay_phai else "phải"
            print(f"L={L} (i={i}, j={j}): lấy trái {nums[i]} -> {lay_trai} | lấy phải {nums[j]} -> {lay_phai} | dp = {dp[i][j]} (chọn {nuoc})")
    print("dp[0][n-1] =", dp[0][n - 1], "->", "Người 1 thắng" if dp[0][n - 1] >= 0 else "Người 1 thua")
    return dp

dp_mau = trace_dp2d([1, 5, 233, 7])
assert dp_mau[0][3] == 222

L=1: đường chéo = [1, 5, 233, 7]
L=2 (i=0, j=1): lấy trái 1 -> -4 | lấy phải 5 -> 4 | dp = 4 (chọn phải)
L=2 (i=1, j=2): lấy trái 5 -> -228 | lấy phải 233 -> 228 | dp = 228 (chọn phải)
L=2 (i=2, j=3): lấy trái 233 -> 226 | lấy phải 7 -> -226 | dp = 226 (chọn trái)
L=3 (i=0, j=2): lấy trái 1 -> -227 | lấy phải 233 -> 229 | dp = 229 (chọn phải)
L=3 (i=1, j=3): lấy trái 5 -> -221 | lấy phải 7 -> -221 | dp = -221 (chọn trái)
L=4 (i=0, j=3): lấy trái 1 -> 222 | lấy phải 7 -> -222 | dp = 222 (chọn trái)
dp[0][n-1] = 222 -> Người 1 thắng


**Trace bảng 1D (Cách 4)** cho cùng mảng, để thấy `dp[j]` đổi nghĩa theo từng hàng `i`:

In [15]:
def trace_dp1d(nums):
    n = len(nums)
    dp = nums[:]
    print("khởi tạo dp =", dp)
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            cu = dp[j]
            dp[j] = max(nums[i] - dp[j], nums[j] - dp[j - 1])
            print(f"i={i}, j={j}: max({nums[i]} - {cu}, {nums[j]} - {dp[j-1]}) = {dp[j]}")
        print(f"  -> sau hàng i={i}: dp = {dp}")
    return dp[n - 1]

assert trace_dp1d([1, 5, 233, 7]) == 222

khởi tạo dp = [1, 5, 233, 7]
i=2, j=3: max(233 - 7, 7 - 233) = 226
  -> sau hàng i=2: dp = [1, 5, 233, 226]
i=1, j=2: max(5 - 233, 233 - 5) = 228
i=1, j=3: max(5 - 226, 7 - 228) = -221
  -> sau hàng i=1: dp = [1, 5, 228, -221]
i=0, j=1: max(1 - 5, 5 - 1) = 4
i=0, j=2: max(1 - 228, 233 - 4) = 229
i=0, j=3: max(1 - -221, 7 - 229) = 222
  -> sau hàng i=0: dp = [1, 4, 229, 222]


## 9. Code nộp LeetCode

Bản dùng Cách 2 (đệ quy có nhớ, ngắn gọn), đúng `class Solution` và chữ ký:

In [16]:
from functools import cache
from typing import List

class Solution:
    def predictTheWinner(self, nums: List[int]) -> bool:
        @cache
        def dfs(i: int, j: int) -> int:
            if i > j:
                return 0
            return max(nums[i] - dfs(i + 1, j), nums[j] - dfs(i, j - 1))

        return dfs(0, len(nums) - 1) >= 0

sol = Solution()
print(sol.predictTheWinner([1, 5, 2]), sol.predictTheWinner([1, 5, 233, 7]))
assert sol.predictTheWinner([1, 5, 2]) is False
assert sol.predictTheWinner([1, 5, 233, 7]) is True

False True


## 10. Vì sao thuật toán đúng

**Cấu trúc con tối ưu.** Trên đoạn `[i..j]`, người đến lượt có đúng hai nước đi. Sau khi lấy `nums[i]`, trò chơi còn lại là **chính trò chơi đó** nhưng trên đoạn `[i+1..j]` và với **vai đổi chỗ**: người kia đi trước. Với giá trị trò chơi `V(i, j)` (hiệu số tối ưu của người đi trước), ta có:

`V(i, j) = max( nums[i] − V(i+1, j), nums[j] − V(i, j−1) )`.

Dấu trừ có mặt vì: nếu đối thủ (đi trước ở đoạn con) đạt hiệu số tối ưu `V(i+1, j)` theo góc nhìn của họ, thì từ góc nhìn của ta nó là `−V(i+1, j)`; và tối ưu của họ (tối đa của họ) chính là tối thiểu của ta (do tổng điểm cố định, trò chơi tổng bằng không). Đây chính xác là minimax gói trong một hàm duy nhất ("negamax").

**Điểm đã tích lũy không ảnh hưởng quyết định:** hiệu số cuối = (hiệu số đã có) + (hiệu số từ đoạn còn lại). Phần đã có là hằng số, nên cực đại/cực tiểu của tổng chỉ phụ thuộc phần còn lại ⇒ Cách 1 và Cách 2 cho cùng giá trị (ta assert điều này bằng số cụ thể, không chỉ `True/False`).

**Đáp án.** Người 1 thắng hoặc hòa ⇔ hiệu số tối ưu `V(0, n−1) ≥ 0`.

**Chứng minh mẹo số chẵn (Cách 5).** Giả sử `n` chẵn. Đặt `E` = tổng các `nums[k]` với `k` chẵn, `O` = tổng với `k` lẻ. Người 1 có chiến lược lấy được **toàn bộ** các phần tử chỉ số chẵn (nếu muốn): lượt đầu mảng là `[0..n−1]`, hai đầu là chỉ số `0` (chẵn) và `n−1` (lẻ) — lấy chỉ số `0`. Giờ còn `[1..n−1]`, **cả hai đầu đều lẻ**, nên Người 2 buộc phải lấy một phần tử lẻ. Sau đó đoạn còn lại có độ dài chẵn và hai đầu khác tính chẵn lẻ (một chẵn, một lẻ) — Người 1 lấy đầu chẵn. Cứ thế, bất biến "đến lượt Người 1, đoạn có độ dài chẵn và đúng một đầu chẵn" được giữ nguyên. Tương tự, Người 1 có thể ép lấy toàn bộ phần tử lẻ. Vậy Người 1 đảm bảo được `max(E, O) ≥ min(E, O)` ≥ phần của Người 2 ⇒ không bao giờ thua.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Minimax thuần | `O(2^n)` | `O(n)` |
| 2. Memo `(i, j)` | `O(n²)` | `O(n²)` |
| 3. DP 2D | `O(n²)` | `O(n²)` |
| 4. DP 1D | `O(n²)` | `O(n)` |
| 5. Mẹo chẵn + DP | chẵn `O(1)`, lẻ `O(n²)` | lẻ `O(n)` |

Với `n ≤ 20`, `n² = 400` thao tác — tức thì. Cách 1 với `n = 20` mất cỡ hàng triệu lần gọi hàm. Hãy thử xem chênh lệch tốc độ trên mảng lẻ dài 19 (để Cách 5 không "ăn gian" bằng mẹo chẵn):

In [17]:
import time, random
random.seed(486)
nums = [random.randint(0, 10 ** 7) for _ in range(19)]
for ten, f in [("vét cạn", chenh_lech_vet_can), ("memo", lambda a: chenh_lech_memo(a)[0]),
               ("dp2d", chenh_lech_dp2d), ("dp1d", chenh_lech_dp1d)]:
    t0 = time.perf_counter()
    kq = f(nums)
    print(f"{ten:8}: hiệu số = {kq:>10}, {time.perf_counter() - t0:.5f}s")

vét cạn : hiệu số =    1779714, 0.11839s
memo    : hiệu số =    1779714, 0.00011s
dp2d    : hiệu số =    1779714, 0.00004s
dp1d    : hiệu số =    1779714, 0.00002s


## 12. Bộ test

Chiến lược: ví dụ đề; biên (`n = 1`, hòa, toàn 0); phản ví dụ của tham lam; **đối chiếu vét cạn** (so cả giá trị hiệu số) với mọi mảng nhỏ; đối chiếu các cách DP với ngẫu nhiên đúng ràng buộc (seed cố định); kiểm tra định lý số chẵn.

In [18]:
import itertools, random

THANG = {
    "vet_can": thang_vet_can,
    "memo": thang_memo,
    "dp2d": thang_dp2d,
    "dp1d": thang_dp1d,
    "chan_le": thang_chan_le,
    "nop": Solution().predictTheWinner,
}

# 1. Ví dụ đề + biên
tests = [([1, 5, 2], False), ([1, 5, 233, 7], True), ([7], True), ([0], True),
         ([1, 1], True), ([0, 0, 0], True), ([3, 9, 1, 2], True), ([2, 4, 55, 6, 8], False)]
for nums, kq in tests:
    for ten, f in THANG.items():
        assert f(nums) == kq, (ten, nums, f(nums), kq)
print("ví dụ + biên: OK")

ví dụ + biên: OK


In [19]:
# 2. Vét cạn (so cả giá trị hiệu số) trên mọi mảng độ dài 1..6 với giá trị 0..3
dem = 0
for L in range(1, 7):
    for nums in itertools.product(range(4), repeat=L):
        nums = list(nums)
        v = chenh_lech_vet_can(nums)
        assert chenh_lech_memo(nums)[0] == v
        assert chenh_lech_dp2d(nums) == v
        assert chenh_lech_dp1d(nums) == v
        assert thang_chan_le(nums) == (v >= 0)
        dem += 1
print("khớp vét cạn (cả giá trị hiệu số) trên", dem, "mảng: OK")

# 3. Ngẫu nhiên có seed, đúng ràng buộc n <= 20, nums[i] <= 10^7
random.seed(2026)
for _ in range(300):
    n = random.randint(1, 20)
    nums = [random.randint(0, 10 ** 7) for _ in range(n)]
    ref = chenh_lech_dp2d(nums)
    assert chenh_lech_memo(nums)[0] == ref
    assert chenh_lech_dp1d(nums) == ref
    if n <= 12:
        assert chenh_lech_vet_can(nums) == ref
    assert thang_chan_le(nums) == (ref >= 0)
print("ngẫu nhiên n <= 20: OK")

khớp vét cạn (cả giá trị hiệu số) trên 5460 mảng: OK
ngẫu nhiên n <= 20: OK


In [20]:
# 4. Định lý số chẵn: n chẵn thì hiệu số tối ưu luôn >= 0
for _ in range(500):
    n = random.choice([2, 4, 6, 8, 10, 12, 20])
    nums = [random.randint(0, 100) for _ in range(n)]
    assert chenh_lech_dp2d(nums) >= 0
print("n chẵn luôn >= 0: OK")

# 5. Điểm của mỗi người từ hiệu số: P1 = (tổng + hiệu số) / 2
nums = [1, 5, 233, 7]
hs = chenh_lech_dp2d(nums)
print("P1 =", (sum(nums) + hs) // 2, "P2 =", (sum(nums) - hs) // 2)
assert (sum(nums) + hs) // 2 == 234 and (sum(nums) - hs) // 2 == 12

n chẵn luôn >= 0: OK
P1 = 234 P2 = 12


**Phản ví dụ cho cách tham lam "luôn lấy đầu lớn hơn":** `nums = [1, 5, 233, 7]`. Tham lam cho `False`, đáp án đúng là `True`.

In [21]:
def tham_lam_sai(nums):
    a = nums[:]
    diem = [0, 0]
    luot = 0
    while a:
        if a[0] >= a[-1]:
            diem[luot] += a.pop(0)
        else:
            diem[luot] += a.pop()
        luot ^= 1
    return diem[0] >= diem[1]

print(tham_lam_sai([1, 5, 233, 7]), Solution().predictTheWinner([1, 5, 233, 7]))
assert tham_lam_sai([1, 5, 233, 7]) != Solution().predictTheWinner([1, 5, 233, 7])

False True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn lấy đầu lớn hơn

Đã thấy ở mục 12: sai với `[1, 5, 233, 7]`. **Vì sao sai:** lấy số lớn nhất trước mắt có thể **mở đường** cho đối thủ ăn số cực lớn bên cạnh. Trò chơi đối kháng cần xét hết các khả năng của đối thủ (minimax).

### Lỗi 2 — Quên đổi dấu ở công thức hiệu số

**Code sai:** `dfs(i, j) = max(nums[i] + dfs(i+1, j), nums[j] + dfs(i, j-1))` (dùng `+` thay `-`).

**Vì sao sai:** `dfs(i+1, j)` là hiệu số theo góc nhìn của **đối thủ**; nó phải **trừ** đi, không cộng.

In [22]:
def sai_dau(nums):
    n = len(nums)
    @lru_cache(maxsize=None)
    def dfs(i, j):
        if i > j:
            return 0
        return max(nums[i] + dfs(i + 1, j), nums[j] + dfs(i, j - 1))     # SAI: dùng +
    return dfs(0, n - 1) >= 0

print([sai_dau(a) for a in ([1, 5, 2], [1, 5, 233, 7])], "đúng:", [False, True])
assert sai_dau([1, 5, 2]) is True           # sai: đúng phải là False

[True, True] đúng: [False, True]


**Code đúng:** dấu `-`.

### Lỗi 3 — Dùng `> 0` thay vì `>= 0` (hòa vẫn thắng)

Đề nói rõ: **hòa tính là Người 1 thắng**. Với `[1, 1]`: hiệu số `0` ⇒ `True`. Viết `> 0` sẽ sai.

In [23]:
def sai_hoa(nums):
    return chenh_lech_dp2d(nums) > 0          # SAI: hòa phải tính là thắng

print(sai_hoa([1, 1]), "đúng:", Solution().predictTheWinner([1, 1]))
assert sai_hoa([1, 1]) is False and Solution().predictTheWinner([1, 1]) is True

False đúng: True


### Lỗi 4 — Khởi tạo bảng sai (quên đường chéo hoặc sao chép hàng)

**Code sai (a):** quên gán `dp[i][i] = nums[i]` nên mọi đoạn một phần tử có hiệu số `0`.

**Code sai (b):** `dp = [[0] * n] * n` (các hàng là cùng một đối tượng).

In [24]:
def sai_khoi_tao(nums):
    n = len(nums)
    dp = [[0] * n for _ in range(n)]          # quên điền đường chéo dp[i][i] = nums[i]
    for L in range(2, n + 1):
        for i in range(0, n - L + 1):
            j = i + L - 1
            dp[i][j] = max(nums[i] - dp[i + 1][j], nums[j] - dp[i][j - 1])
    return dp[0][n - 1]

print(sai_khoi_tao([1, 5, 2]), "đúng:", chenh_lech_dp2d([1, 5, 2]))
assert sai_khoi_tao([1, 5, 2]) != chenh_lech_dp2d([1, 5, 2])

-3 đúng: -2


**Mẹo tránh lỗi:** luôn kiểm tra đường chéo (đoạn dài 1) và một ví dụ có `n = 3`, `n = 4` bằng tay trước khi nộp.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** hai người luân phiên, cả hai tối ưu, lấy từ hai đầu đoạn / trạng thái là đoạn liên tiếp ⇒ **DP theo đoạn** `dp[i][j]`.
- **Trò chơi tổng bằng không** ⇒ quy về **hiệu số** và dùng `a − dp[...]` (negamax). Không cần theo dõi hai điểm riêng biệt.
- **Muốn chắc đáp án:** viết thêm một minimax vét cạn (có điểm số tường minh) làm oracle.
- **Nhớ:** `Người 1 thắng` khi hiệu số `>= 0`.
- **Phát hiện tính chất đặc biệt:** thử vét cạn mảng nhỏ rồi quan sát. Với `n` chẵn ta thấy đáp án luôn `True` (Stone Game 877).
- Khi DP 2D chỉ dùng hàng kế tiếp, chuyển sang **mảng 1D** để tiết kiệm bộ nhớ.

## 15. Biến thể

1. **Điểm tối đa của Người 1:** `(sum(nums) + hiệu số) // 2`.
2. **Stone Game (877):** số đống chẵn, tổng lẻ ⇒ Người 1 luôn thắng (cùng chứng minh tính chẵn lẻ).
3. **Stone Game II/III/VII (1140, 1406, 1690):** cùng khung "hiệu số / negamax", đổi luật lấy hoặc cách tính điểm.
4. **Can I Win (464):** trạng thái là tập số đã chọn ⇒ bitmask + memo.
5. **Trò chơi nhiều hơn hai người hoặc không tổng bằng không:** công thức `a − dp` không còn đúng, cần theo dõi vector điểm.

In [25]:
def diem_toi_da_nguoi_1(nums):
    hs = chenh_lech_dp2d(nums)
    return (sum(nums) + hs) // 2

print(diem_toi_da_nguoi_1([1, 5, 2]), diem_toi_da_nguoi_1([1, 5, 233, 7]))    # 3 234
assert diem_toi_da_nguoi_1([1, 5, 2]) == 3 and diem_toi_da_nguoi_1([1, 5, 233, 7]) == 234

# Stone Game 877: số đống chẵn, tổng lẻ, đáp án luôn True
random.seed(877)
for _ in range(200):
    n = random.choice([2, 4, 6, 8])
    piles = random.sample(range(1, 200), n)
    if sum(piles) % 2 == 1:
        assert chenh_lech_dp2d(piles) > 0
print("Stone Game: OK")

3 234
Stone Game: OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax thuần | thử mọi nước đi, theo dõi hai điểm số | `O(2^n)` | `O(n)` | oracle cho `n ≲ 14` |
| 2. Memo `(i,j)` | hiệu số negamax + bộ nhớ đệm | `O(n²)` | `O(n²)` | dễ viết nhất, nên nộp |
| 3. DP 2D | điền bảng theo độ dài đoạn | `O(n²)` | `O(n²)` | khi muốn tránh đệ quy |
| 4. DP 1D | cuốn chiếu hàng `i` | `O(n²)` | `O(n)` | tiết kiệm bộ nhớ |
| 5. Mẹo chẵn | `n` chẵn ⇒ `True` | `O(1)` chẵn | — | tăng tốc, hiểu cấu trúc trò chơi |

**Khuyên dùng khi thi:** Cách 2 (hoặc Cách 4 nếu bị yêu cầu bộ nhớ nhỏ).

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `chenh_lech_dp2d([3, 9])`, `chenh_lech_dp2d([2, 2, 2])` và `thang_dp1d([4, 2])`; sau đó chạy để kiểm tra.

In [26]:
du_doan_1 = None
du_doan_2 = None
du_doan_3 = None
print("Thực tế:", chenh_lech_dp2d([3, 9]), chenh_lech_dp2d([2, 2, 2]), thang_dp1d([4, 2]))

Thực tế: 6 2 True


**Bài 2 (tính tay).** Với `nums = [3, 9, 1, 2]` hãy điền bảng `dp[i][j]` bằng tay (đường chéo, độ dài 2, độ dài 3, độ dài 4) rồi nói Người 1 có thắng không và điểm mỗi người là bao nhiêu.

**Bài 3 (điền TODO).** Hoàn thiện `dp2d_todo(nums)` trả về `dp[0][n-1]` bằng bảng 2D.

In [27]:
def dp2d_todo(nums):
    n = len(nums)
    # TODO 1: tạo bảng dp n x n bằng [[0] * n for _ in range(n)]
    # TODO 2: điền dp[i][i] = nums[i]
    # TODO 3: for L in range(2, n + 1): for i in range(n - L + 1): j = i + L - 1;
    #         dp[i][j] = max(nums[i] - dp[i+1][j], nums[j] - dp[i][j-1])
    # TODO 4: return dp[0][n-1]
    pass

kq_bai3 = None        # sau khi viết xong: kq_bai3 = dp2d_todo([3, 9, 1, 2])

**Bài 4 (tự viết lại).** Không nhìn lời giải, tự viết `thang_tu_viet(nums)` bằng đệ quy có nhớ (`@lru_cache`).

In [28]:
def thang_tu_viet(nums):
    # TODO: tự viết
    pass

**Bài 5 (biến thể / mini challenge).** Mỗi lượt, người chơi có thể lấy **1 hoặc 2** số liên tiếp ở **đầu bên trái** hoặc bên phải (tức lấy 1 số hoặc 2 số ở cùng một phía). Viết `thang_lay_1_hoac_2(nums)` trả về `True` nếu Người 1 thắng hoặc hòa. Gợi ý: `dfs(i, j)` thêm hai lựa chọn lấy hai số nếu đoạn còn ít nhất hai phần tử.

In [29]:
def thang_lay_1_hoac_2(nums):
    # TODO: viết tự do
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):**
  - Bài 1: với hai phần tử `[a, b]` hiệu số là `max(a - b, b - a) = |a - b|`.
  - Bài 3: dùng đúng các bước TODO.
  - Bài 5: thêm hai nước đi: lấy `nums[i] + nums[i+1]` hoặc `nums[j-1] + nums[j]`.
- **Mức 2 (vừa):**
  - Bài 2: `dp[0][1] = 6`, `dp[1][2] = 8`, `dp[2][3] = 1`; sau đó tính các đoạn ba phần tử.
  - Bài 4: hàm trong có `@lru_cache(maxsize=None)`; trả `dfs(0, n-1) >= 0`.
  - Bài 5: `dfs(i, j) = max(nums[i] - dfs(i+1, j), nums[j] - dfs(i, j-1), nums[i] + nums[i+1] - dfs(i+2, j), nums[j-1] + nums[j] - dfs(i, j-2))`, nhớ đặt điều kiện `i + 1 <= j` cho hai nước lấy-hai-số, và `dfs = 0` khi `i > j`.
- **Mức 3 (gần đáp án):**
  - Bài 2: `dp[0][2] = max(3 - 8, 1 - 6) = -5`; `dp[1][3] = max(9 - 1, 2 - 8) = 8`; `dp[0][3] = max(3 - 8, 2 - (-5)) = 7`.
  - Bài 4: copy Cách 2.
  - Bài 5: kiểm tra bằng vét cạn nhỏ tương tự (viết minimax có hai điểm số cho luật mới).

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert chenh_lech_dp2d([3, 9]) == 6
assert chenh_lech_dp2d([2, 2, 2]) == 2
assert thang_dp1d([4, 2]) is True
print(chenh_lech_dp2d([3, 9]), chenh_lech_dp2d([2, 2, 2]), thang_dp1d([4, 2]))

# Bài 2: [3, 9, 1, 2]
assert chenh_lech_dp2d([3, 9, 1, 2]) == 7
assert (sum([3, 9, 1, 2]) + 7) // 2 == 11 and (sum([3, 9, 1, 2]) - 7) // 2 == 4
print("Bài 2: P1 = 11, P2 = 4")

6 2 True
Bài 2: P1 = 11, P2 = 4


In [31]:
# Bài 3
def dp2d_dap_an(nums):
    n = len(nums)
    dp = [[0] * n for _ in range(n)]
    for i in range(n):
        dp[i][i] = nums[i]
    for L in range(2, n + 1):
        for i in range(n - L + 1):
            j = i + L - 1
            dp[i][j] = max(nums[i] - dp[i + 1][j], nums[j] - dp[i][j - 1])
    return dp[0][n - 1]

assert dp2d_dap_an([3, 9, 1, 2]) == 7
for nums in itertools.product(range(4), repeat=5):
    assert dp2d_dap_an(list(nums)) == chenh_lech_vet_can(list(nums))

# Bài 4
def thang_tu_viet_dap_an(nums):
    @lru_cache(maxsize=None)
    def dfs(i, j):
        if i > j:
            return 0
        return max(nums[i] - dfs(i + 1, j), nums[j] - dfs(i, j - 1))
    return dfs(0, len(nums) - 1) >= 0

for nums, kq in tests:
    assert thang_tu_viet_dap_an(nums) == kq

In [32]:
# Bài 5: lấy 1 hoặc 2 số liên tiếp ở MỘT đầu
def thang_lay_1_hoac_2_dap_an(nums):
    n = len(nums)

    @lru_cache(maxsize=None)
    def dfs(i, j):
        if i > j:
            return 0
        best = max(nums[i] - dfs(i + 1, j), nums[j] - dfs(i, j - 1))
        if i + 1 <= j:
            best = max(best, nums[i] + nums[i + 1] - dfs(i + 2, j),
                       nums[j - 1] + nums[j] - dfs(i, j - 2))
        return best
    return dfs(0, n - 1) >= 0

# oracle minimax có hai điểm số tường minh cho luật mới
def oracle_bai5(nums):
    def tran(l, r, d1, d2, luot):
        if l > r:
            return d1 - d2
        kq = []
        for k in (1, 2):
            if r - l + 1 >= k:
                gl = sum(nums[l:l + k])
                gr = sum(nums[r - k + 1:r + 1])
                if luot == 1:
                    kq.append(tran(l + k, r, d1 + gl, d2, 2))
                    kq.append(tran(l, r - k, d1 + gr, d2, 2))
                else:
                    kq.append(tran(l + k, r, d1, d2 + gl, 1))
                    kq.append(tran(l, r - k, d1, d2 + gr, 1))
        return max(kq) if luot == 1 else min(kq)
    return tran(0, len(nums) - 1, 0, 0, 1) >= 0

random.seed(5)
for _ in range(150):
    nums = [random.randint(0, 9) for _ in range(random.randint(1, 8))]
    assert thang_lay_1_hoac_2_dap_an(nums) == oracle_bai5(nums), nums
print("Tất cả đáp án OK")

Tất cả đáp án OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (negamax có nhớ trên đoạn):**

```python
from functools import cache
from typing import List

class Solution:
    def predictTheWinner(self, nums: List[int]) -> bool:
        @cache
        def dfs(i: int, j: int) -> int:
            if i > j:
                return 0
            return max(nums[i] - dfs(i + 1, j), nums[j] - dfs(i, j - 1))

        return dfs(0, len(nums) - 1) >= 0
```

**Checklist trước khi submit:**

- [ ] `dfs(i, j)` là hiệu số của **người đang đi** trên đoạn `[i..j]`; công thức dùng dấu **trừ** `nums[i] - dfs(i+1, j)`.
- [ ] Cơ sở: `i > j` ⇒ `0`.
- [ ] Đáp án: `dfs(0, n-1) >= 0` (hòa vẫn thắng).
- [ ] Nếu làm bảng: tạo bằng `[[0] * n for _ in range(n)]`, điền đường chéo, duyệt theo độ dài tăng dần.
- [ ] Không dùng tham lam "lấy số lớn hơn".
- [ ] Đã thử `[1,5,2]` (False), `[1,5,233,7]` (True), `[1,1]` (True), `n = 1`.
- [ ] Độ phức tạp `O(n²)` thời gian, `O(n²)` (hoặc `O(n)`) bộ nhớ.